# DA2 Metric-Outdoor-Large — Unified Evaluation Pipeline *(Pre-Masked Edition)*

Evaluates **Depth-Anything-V2-Metric-Outdoor-Large** on the **KITTI Eigen-Split** dataset.

**Key change vs. original `DA_2_metric_analysis_claude.py`:**  
Binary instance masks are **loaded from the pre-generated mask store** produced by  
`inst_seg_notebook.ipynb` — SAM3 is **never** called at runtime.

### Pipeline
| Step | What happens |
|------|-------------|
| 1 | Index KITTI Eigen-Split (test + optional train) |
| 2 | Load DA2 Metric-Outdoor-Large depth model (HuggingFace) |
| 3 | For each frame: load masks → run DA2 depth inference → compute scale variants |
| 4 | Write per-mask metrics to CSV |
| 5 | Generate 12 analysis plots |

**Mask store layout** (mirrors KITTI root):
```
MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
    <frame_stem>_manifest.json
    <frame_stem>_<super_cat>_<sub_cat>_<local_idx>_<uid>_<score>_mask.png
```

**Model reference:**  
https://github.com/DepthAnything/Depth-Anything-V2/tree/main/metric_depth


## 0 · GPU Selection & Imports

In [15]:
# # ── GPU selection ────────────────────────────────────────────────────────────
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # ← edit as needed

# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import glob
import json
import math
import logging
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D          # noqa: F401
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# NOTE: seg_code / SAM3 imports removed — masks are loaded from disk.
print("Imports OK  |  CUDA available:", torch.cuda.is_available())


Imports OK  |  CUDA available: True


## 1 · Configuration  *(edit paths here)*

In [16]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── KITTI paths ───────────────────────────────────────────────────────────────
KITTI_ROOT  = "/STORAGE/Arka/Downloads/KITTI/kitti_eigen_split_dataset_test"
TEST_ROOT   = os.path.join(KITTI_ROOT, "test")
TRAIN_ROOT  = os.path.join(KITTI_ROOT, "train")
USE_TRAIN   = False   # set True to also process the train split

# ── Pre-generated mask store ──────────────────────────────────────────────────
# Root produced by inst_seg_notebook.ipynb.
# Sub-tree under MASK_ROOT mirrors KITTI_ROOT exactly:
#   MASK_ROOT / <split> / <date> / <drive> / image_02/data / <frame_stem> /
MASK_ROOT   = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output"

# ── Depth bounds (KITTI standard) ─────────────────────────────────────────────
MIN_DEPTH   = 1e-3
MAX_DEPTH   = 80.0

# ── KITTI focal length ────────────────────────────────────────────────────────
FOCAL_LENGTH_X = 721.5577

# ── Depth Anything V2 — metric, outdoor, Large ───────────────────────────────
# HuggingFace model ID (metric variant):
#   https://github.com/DepthAnything/Depth-Anything-V2/tree/main/metric_depth
DA2_CHECKPOINT = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf"
HF_CACHE_DIR   = "/STORAGE/Arka/cache"

# ── Mask confidence gate ──────────────────────────────────────────────────────
MIN_MASK_SCORE = 0.50   # masks below this SAM3 score are ignored

# ── Output directories ────────────────────────────────────────────────────────
MAIN_OUTPUT_DIR = "/STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/DA2_Metric_Large_outputs"
OUTPUT_CSV_PATH = os.path.join(MAIN_OUTPUT_DIR, "da2_metric_results.csv")
PLOT_DIR        = os.path.join(MAIN_OUTPUT_DIR, "plots")
NPY_DIR         = os.path.join(MAIN_OUTPUT_DIR, "npy_cache")
SAVE_NPY        = True   # save raw DA2 depth arrays as .npy for later inspection

# ── Resume: set > 0 to hard-skip to a frame index ────────────────────────────
START_INDEX = 0

print("Configuration loaded.")


Configuration loaded.


## 2 · Logging

In [17]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("da2_metric_eval")
logging.getLogger("transformers").setLevel(logging.ERROR)


## 3 · Metric & Crop Helpers

In [18]:
def eigen_crop_mask(shape) -> np.ndarray:
    """Boolean mask for the standard KITTI Eigen evaluation crop."""
    h, w  = shape[:2]
    m     = np.zeros((h, w), dtype=bool)
    top   = 44
    bot   = min(top + 352, h)
    m[top:bot, 0:w] = True
    return m


def compute_depth_errors(gt: np.ndarray,
                         pred: np.ndarray,
                         mask: np.ndarray) -> dict:
    """Standard KITTI / Monodepth2 depth metrics (masked pixels only)."""
    gt_v   = gt[mask]
    pred_v = pred[mask]
    pred_v = np.maximum(pred_v, 1e-8)

    thresh   = np.maximum(gt_v / pred_v, pred_v / gt_v)
    rmse     = float(np.sqrt(np.mean((gt_v - pred_v) ** 2)))
    abs_rel  = float(np.mean(np.abs(gt_v - pred_v) / gt_v))
    sq_rel   = float(np.mean(((gt_v - pred_v) ** 2) / gt_v))
    rmse_log = float(np.sqrt(np.mean((np.log(gt_v) - np.log(pred_v)) ** 2)))
    delta1   = float(np.mean(thresh < 1.25))
    delta2   = float(np.mean(thresh < 1.25 ** 2))
    delta3   = float(np.mean(thresh < 1.25 ** 3))
    return dict(rmse=rmse, abs_rel=abs_rel, sq_rel=sq_rel,
                rmse_log=rmse_log, delta1=delta1, delta2=delta2, delta3=delta3)

print("Metric helpers defined.")


Metric helpers defined.


## 4 · Scale Computation

In [19]:
def compute_all_scales(gt_depth: np.ndarray,
                       pred_raw: np.ndarray,
                       mask: np.ndarray) -> dict:
    """
    Compute four scale-aligned depth maps from the same raw prediction.

    Returns a dict keyed by scale name ('unscaled', 'median', 'mean', 'affine').
    Each value contains: scale, shift, depth_map, avg_depth, min_depth,
    and all seven depth-error metrics.
    """
    gt_v   = gt_depth[mask]
    pred_c = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)
    out    = {}

    # ── No scaling ────────────────────────────────────────────────────────────
    m0 = compute_depth_errors(gt_depth, pred_c, mask)
    out["unscaled"] = {
        "scale": 1.0, "shift": 0.0, "depth_map": pred_c, **m0,
        "avg_depth": float(np.mean(pred_c[mask])),
        "min_depth": float(np.min(pred_c[mask])),
    }

    # ── Median scale (ratio of medians) ───────────────────────────────────────
    scale_med = float(np.median(gt_v) / np.median(pred_c[mask]))
    pred_med  = np.clip(pred_c * scale_med, MIN_DEPTH, MAX_DEPTH)
    m1 = compute_depth_errors(gt_depth, pred_med, mask)
    out["median"] = {
        "scale": scale_med, "shift": 0.0, "depth_map": pred_med, **m1,
        "avg_depth": float(np.mean(pred_med[mask])),
        "min_depth": float(np.min(pred_med[mask])),
    }

    # ── Mean scale (ratio of means) ───────────────────────────────────────────
    scale_mean = float(np.mean(gt_v) / np.mean(pred_c[mask]))
    pred_mean  = np.clip(pred_c * scale_mean, MIN_DEPTH, MAX_DEPTH)
    m2 = compute_depth_errors(gt_depth, pred_mean, mask)
    out["mean"] = {
        "scale": scale_mean, "shift": 0.0, "depth_map": pred_mean, **m2,
        "avg_depth": float(np.mean(pred_mean[mask])),
        "min_depth": float(np.min(pred_mean[mask])),
    }

    # ── Affine scale (OLS: scale * pred + shift ≈ GT) ─────────────────────────
    pred_flat       = pred_c[mask].ravel()
    A               = np.vstack([pred_flat, np.ones_like(pred_flat)]).T
    params, _, _, _ = np.linalg.lstsq(A, gt_v.ravel(), rcond=None)
    scale_aff       = float(params[0])
    shift_aff       = float(params[1])
    pred_aff        = np.clip(scale_aff * pred_c + shift_aff, MIN_DEPTH, MAX_DEPTH)
    m3 = compute_depth_errors(gt_depth, pred_aff, mask)
    out["affine"] = {
        "scale": scale_aff, "shift": shift_aff, "depth_map": pred_aff, **m3,
        "avg_depth": float(np.mean(pred_aff[mask])),
        "min_depth": float(np.min(pred_aff[mask])),
    }

    return out

print("Scale computation functions defined.")


Scale computation functions defined.


## 5 · DA2 Metric-Outdoor-Large — Model Loading & Inference

Uses **`AutoImageProcessor`** + **`AutoModelForDepthEstimation`** from HuggingFace  
`transformers`.  The metric model outputs depth directly in **metres** — no  
inversion or global scale factor is applied.


In [20]:
def load_da2(checkpoint: str = DA2_CHECKPOINT,
             cache_dir:  str = HF_CACHE_DIR) -> tuple:
    """Load Depth-Anything-V2-Metric-Outdoor-Large from HuggingFace."""
    log.info("Loading Depth-Anything-V2-Metric-Outdoor-Large …")
    proc = AutoImageProcessor.from_pretrained(
        checkpoint, cache_dir=cache_dir, use_fast=True
    )
    mdl  = AutoModelForDepthEstimation.from_pretrained(
        checkpoint, cache_dir=cache_dir
    ).to(DEVICE).eval()
    log.info("DA2 Metric model ready on %s.", DEVICE)
    return proc, mdl


def run_da2(image_np: np.ndarray, proc, mdl) -> np.ndarray:
    """
    Run Depth-Anything-V2-Metric on one RGB uint8 numpy array (H×W×3).

    Returns a metric depth map in metres, same H×W as input.
    The model output is already in metres — no inversion is applied.
    """
    img_pil = Image.fromarray(image_np)
    inputs  = proc(images=img_pil, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        raw = mdl(**inputs).predicted_depth    # shape: (1, H', W')

    depth = F.interpolate(
        raw.unsqueeze(1),
        size=img_pil.size[::-1],               # (H, W)
        mode="bicubic",
        align_corners=False,
    ).squeeze().cpu().numpy()

    return depth.astype(np.float32)            # metres, float32

print("DA2 model helpers defined.")


DA2 model helpers defined.


## 6 · Pre-Generated Mask Loading  *(replaces SAM3 inference)*

The two functions below resolve the mask directory for any KITTI frame and
load all pre-generated instance masks from it.  The manifest JSON is used
when available (fast path); otherwise the file names are parsed directly.


In [21]:
def get_mask_dir(rgb_path: str) -> str:
    """
    Derive the pre-generated mask directory for a given KITTI RGB image path.

    Mapping:
        KITTI_ROOT / <split> / <date> / <drive> / image_02/data / <frame>.png
           →
        MASK_ROOT  / <split> / <date> / <drive> / image_02/data / <frame_stem> /

    Parameters
    ----------
    rgb_path : Absolute path to the KITTI RGB image.

    Returns
    -------
    Absolute path to the frame-level mask directory (may not exist if the
    frame was skipped during mask generation).
    """
    frame_stem   = os.path.splitext(os.path.basename(rgb_path))[0]
    rel_data_dir = os.path.relpath(os.path.dirname(rgb_path), KITTI_ROOT)
    # e.g. "test/2011_09_26/2011_09_26_drive_0001_sync/image_02/data"
    return os.path.join(MASK_ROOT, rel_data_dir, frame_stem)


def load_masks_for_frame(mask_dir: str,
                          super_cat_filter: str = None,
                          sub_cat_filter:   str = None) -> list:
    """
    Load all pre-generated instance masks for one KITTI frame.

    Prefers the JSON manifest (fast path).  Falls back to glob + filename
    parsing when the manifest is absent.

    Parameters
    ----------
    mask_dir         : Absolute path to the frame-level mask directory.
    super_cat_filter : If given, return only instances of this super-category.
    sub_cat_filter   : If given, return only instances of this sub-category.

    Returns
    -------
    List of dicts, each containing:
        uid, local_idx, super_cat, sub_cat, score, box,
        mask (H×W bool ndarray), path (Path object).
    Returns an empty list when the directory doesn't exist or has no masks.
    """
    mask_dir_path = Path(mask_dir)
    if not mask_dir_path.exists():
        return []

    frame_stem    = mask_dir_path.name
    manifest_path = mask_dir_path / f"{frame_stem}_manifest.json"

    # ── Fast path: use manifest ───────────────────────────────────────────────
    if manifest_path.exists():
        with open(manifest_path, "r") as fh:
            records = json.load(fh)

        instances = []
        for rec in records:
            if super_cat_filter and rec["super_cat"] != super_cat_filter:
                continue
            if sub_cat_filter and rec["sub_cat"] != sub_cat_filter:
                continue
            mask_file = mask_dir_path / rec["mask_file"]
            if not mask_file.exists():
                log.warning("Mask file missing: %s", mask_file)
                continue
            mask_np = np.array(Image.open(mask_file).convert("L")) > 127
            instances.append({
                "uid":       rec["uid"],
                "local_idx": rec["local_idx"],
                "super_cat": rec["super_cat"],
                "sub_cat":   rec["sub_cat"],
                "score":     float(rec["score"]),
                "box":       rec.get("box"),
                "mask":      mask_np,
                "path":      mask_file,
            })
        return instances

    # ── Fallback: glob + filename parsing ─────────────────────────────────────
    # Filename format produced by inst_seg_notebook.ipynb:
    #   <frame_stem>_<super_cat>_<sub_cat>_<local_idx:04d>_<uid8>_<score:.3f>_mask.png
    instances = []
    for mask_file in sorted(mask_dir_path.glob("*_mask.png")):
        stem = mask_file.stem[:-5]           # strip "_mask"
        if not stem.startswith(frame_stem + "_"):
            continue
        suffix = stem[len(frame_stem) + 1:]

        parts = suffix.rsplit("_", 3)        # [categories, local_idx, uid, score]
        if len(parts) != 4:
            log.warning("Unexpected mask filename format: %s", mask_file.name)
            continue
        cat_str, idx_str, uid_str, score_str = parts

        cat_parts = cat_str.split("_", 1)
        if len(cat_parts) != 2:
            log.warning("Cannot parse categories from: %s", cat_str)
            continue
        super_cat, sub_cat = cat_parts

        if super_cat_filter and super_cat != super_cat_filter:
            continue
        if sub_cat_filter and sub_cat != sub_cat_filter:
            continue

        try:
            local_idx = int(idx_str)
            score     = float(score_str)
        except ValueError:
            log.warning("Cannot parse idx/score from: %s", mask_file.name)
            continue

        mask_np = np.array(Image.open(mask_file).convert("L")) > 127
        instances.append({
            "uid":       uid_str,
            "local_idx": local_idx,
            "super_cat": super_cat,
            "sub_cat":   sub_cat,
            "score":     score,
            "box":       None,
            "mask":      mask_np,
            "path":      mask_file,
        })

    instances.sort(key=lambda x: x["local_idx"])
    return instances

print("Mask loading helpers defined.")


Mask loading helpers defined.


## 7 · KITTI Dataset Indexing

In [22]:
def index_kitti(use_train: bool = USE_TRAIN) -> pd.DataFrame:
    """
    Walk KITTI Eigen-Split directories and build a DataFrame of
    (RGB path, GT depth path) pairs for test (and optionally train) frames.
    """
    rows = []

    def _collect(split_root: str, tag: str) -> None:
        pattern  = os.path.join(
            split_root, "*", "*_sync",
            "proj_depth", "groundtruth", "image_02", "*.png"
        )
        gt_paths = glob.glob(pattern)
        for gt_path in tqdm(gt_paths, desc=f"Indexing {tag}"):
            parts              = gt_path.split(os.sep)
            date, drive, fname = parts[-6], parts[-5], parts[-1]
            rgb_path = os.path.join(split_root, date, drive,
                                    "image_02", "data", fname)
            if not os.path.exists(rgb_path):
                continue
            rows.append(dict(
                folder=tag, sequence=date, drive=drive,
                filename=fname, gt_path=gt_path, rgb_path=rgb_path,
            ))

    _collect(TEST_ROOT,  "Test")
    if use_train:
        _collect(TRAIN_ROOT, "Train")

    df = pd.DataFrame(rows).reset_index(drop=True)
    log.info("Dataset indexed: %d frames, %d drives.",
             len(df), df["drive"].nunique())
    return df

print("KITTI indexing function defined.")


KITTI indexing function defined.


## 8 · CSV Column Schema

In [23]:
CSV_COLUMNS = [
    # ── Frame & mask identity ─────────────────────────────────────────────────
    "img_id",             # integer row index in the dataset DataFrame
    "file_name",          # unique string: folder_drive_frameid
    "mask_id",            # local_idx of this mask within the frame
    "mask_uid",           # globally unique mask UID from SAM3 manifest
    "mask_score",         # SAM3 detection confidence
    "super_cat",          # super-category (vehicle / pedestrian / stationary_object)
    "sub_cat",            # sub-category
    "horizontal_angle",   # degrees from image centre axis

    # ── LiDAR ground truth (mask-region only) ─────────────────────────────────
    "lidar_avg_depth",    # mean GT depth over valid mask pixels (metres)
    "lidar_min_depth",    # min  GT depth over valid mask pixels (metres)

    # ── Unscaled depth ────────────────────────────────────────────────────────
    "unscaled_rmse",      "unscaled_abs_rel",   "unscaled_delta1",
    "unscaled_avg_depth", "unscaled_min_depth",

    # ── Median scale ──────────────────────────────────────────────────────────
    "median_scale",
    "median_rmse",        "median_abs_rel",     "median_delta1",
    "median_avg_depth",   "median_min_depth",

    # ── Mean scale ────────────────────────────────────────────────────────────
    "mean_scale",
    "mean_rmse",          "mean_abs_rel",       "mean_delta1",
    "mean_avg_depth",     "mean_min_depth",

    # ── Affine scale ──────────────────────────────────────────────────────────
    "affine_scale",       "affine_shift",
    "affine_rmse",        "affine_abs_rel",     "affine_delta1",
    "affine_avg_depth",   "affine_min_depth",

    # ── File paths ────────────────────────────────────────────────────────────
    "rgb_path",
    "gt_depth_path",
    "da2_pred_path",      # path to saved .npy depth array (or "" if SAVE_NPY=False)
]
print(f"CSV schema: {len(CSV_COLUMNS)} columns defined.")


CSV schema: 37 columns defined.


## 9 · Main Processing Loop

In [24]:
def process_dataset(da2_proc, da2_mdl, val_list: pd.DataFrame) -> None:
    """
    For each KITTI frame:
      1. Load pre-generated masks from MASK_ROOT.
      2. Run DA2 Metric inference.
      3. For each mask, compute four scale variants and all error metrics.
      4. Append results to OUTPUT_CSV_PATH (append mode — safe to resume).
    """
    os.makedirs(MAIN_OUTPUT_DIR, exist_ok=True)
    os.makedirs(NPY_DIR,         exist_ok=True)

    # ── Initialise / resume CSV ───────────────────────────────────────────────
    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(OUTPUT_CSV_PATH, index=False)

    processed_files: set = set()
    if os.path.exists(OUTPUT_CSV_PATH):
        prev = pd.read_csv(OUTPUT_CSV_PATH)
        if "file_name" in prev.columns:
            processed_files = set(prev["file_name"].dropna().unique())

    frames_skipped_no_masks = 0
    total = len(val_list)

    for idx, row in tqdm(val_list.iterrows(), total=total,
                         desc="Processing frames", dynamic_ncols=True):
        if idx < START_INDEX:
            continue

        rgb_path  = row["rgb_path"]
        gt_path   = row["gt_path"]
        folder    = row["folder"]
        drive     = row["drive"]
        filename  = row["filename"]
        stem      = os.path.splitext(filename)[0]
        file_name = f"{folder}_{drive}_{stem}"

        if file_name in processed_files:
            continue

        try:
            # ── Load masks ────────────────────────────────────────────────────
            mask_dir  = get_mask_dir(rgb_path)
            instances = load_masks_for_frame(mask_dir)

            if not instances:
                frames_skipped_no_masks += 1
                continue

            # ── Load RGB + GT depth ───────────────────────────────────────────
            image_np = np.array(Image.open(rgb_path).convert("RGB"))
            gt_raw   = np.array(Image.open(gt_path)).astype(np.uint16)
            gt_depth = gt_raw / 256.0            # KITTI uint16 → metres

            img_h, img_w = image_np.shape[:2]
            principal_x  = img_w / 2.0

            # ── Run DA2 Metric ────────────────────────────────────────────────
            pred_raw  = run_da2(image_np, da2_proc, da2_mdl)
            pred_raw  = np.clip(pred_raw, MIN_DEPTH, MAX_DEPTH)

            # ── Optionally save depth array ───────────────────────────────────
            da2_pred_path = ""
            if SAVE_NPY:
                npy_path = os.path.join(NPY_DIR, f"{file_name}_da2.npy")
                np.save(npy_path, pred_raw)
                da2_pred_path = npy_path

            # ── Eigen crop ────────────────────────────────────────────────────
            crop = eigen_crop_mask(gt_depth.shape)

            # ── Per-mask evaluation ───────────────────────────────────────────
            rows_out = []
            for inst in instances:
                score     = inst["score"]
                super_cat = inst["super_cat"]
                sub_cat   = inst["sub_cat"]
                local_idx = inst["local_idx"]
                uid       = inst["uid"]
                mask_raw  = inst["mask"]

                if score < MIN_MASK_SCORE:
                    continue

                # Resize mask if dimensions differ from the current image
                if mask_raw.shape != (img_h, img_w):
                    mask_pil = Image.fromarray(mask_raw.astype(np.uint8) * 255)
                    mask_pil = mask_pil.resize((img_w, img_h), Image.NEAREST)
                    mask_raw = np.array(mask_pil) > 127

                # Combined validity: instance mask ∩ valid GT ∩ Eigen crop
                valid = (
                    mask_raw
                    & (gt_depth > MIN_DEPTH)
                    & (gt_depth < MAX_DEPTH)
                    & crop
                )
                if not np.any(valid):
                    continue

                # ── LiDAR (GT) stats in mask region ──────────────────────────
                gt_vals   = gt_depth[valid]
                lidar_avg = float(np.mean(gt_vals))
                lidar_min = float(np.min(gt_vals))

                # ── Horizontal angle of mask centroid ─────────────────────────
                ys, xs  = np.where(mask_raw)
                cent_x  = float(np.mean(xs)) if len(xs) > 0 else principal_x
                angle   = math.degrees(
                    math.atan((cent_x - principal_x) / FOCAL_LENGTH_X)
                )

                # ── Four scale variants ───────────────────────────────────────
                scales = compute_all_scales(gt_depth, pred_raw, valid)
                u = scales["unscaled"]
                m = scales["median"]
                n = scales["mean"]
                a = scales["affine"]

                rows_out.append({
                    # Identity
                    "img_id":           idx,
                    "file_name":        file_name,
                    "mask_id":          local_idx,
                    "mask_uid":         uid,
                    "mask_score":       score,
                    "super_cat":        super_cat,
                    "sub_cat":          sub_cat,
                    "horizontal_angle": angle,

                    # LiDAR reference
                    "lidar_avg_depth":  lidar_avg,
                    "lidar_min_depth":  lidar_min,

                    # Unscaled
                    "unscaled_rmse":      u["rmse"],
                    "unscaled_abs_rel":   u["abs_rel"],
                    "unscaled_delta1":    u["delta1"],
                    "unscaled_avg_depth": u["avg_depth"],
                    "unscaled_min_depth": u["min_depth"],

                    # Median scale
                    "median_scale":      m["scale"],
                    "median_rmse":       m["rmse"],
                    "median_abs_rel":    m["abs_rel"],
                    "median_delta1":     m["delta1"],
                    "median_avg_depth":  m["avg_depth"],
                    "median_min_depth":  m["min_depth"],

                    # Mean scale
                    "mean_scale":        n["scale"],
                    "mean_rmse":         n["rmse"],
                    "mean_abs_rel":      n["abs_rel"],
                    "mean_delta1":       n["delta1"],
                    "mean_avg_depth":    n["avg_depth"],
                    "mean_min_depth":    n["min_depth"],

                    # Affine scale
                    "affine_scale":      a["scale"],
                    "affine_shift":      a["shift"],
                    "affine_rmse":       a["rmse"],
                    "affine_abs_rel":    a["abs_rel"],
                    "affine_delta1":     a["delta1"],
                    "affine_avg_depth":  a["avg_depth"],
                    "affine_min_depth":  a["min_depth"],

                    # Paths
                    "rgb_path":      rgb_path,
                    "gt_depth_path": gt_path,
                    "da2_pred_path": da2_pred_path,
                })

            if rows_out:
                pd.DataFrame(rows_out, columns=CSV_COLUMNS).to_csv(
                    OUTPUT_CSV_PATH, mode="a", header=False, index=False
                )
                processed_files.add(file_name)

        except Exception as exc:
            log.error("Error on %s: %s", file_name, exc, exc_info=True)

    log.info("Processing complete → %s", OUTPUT_CSV_PATH)
    if frames_skipped_no_masks > 0:
        log.warning(
            "%d frames skipped (no pre-generated masks in MASK_ROOT).",
            frames_skipped_no_masks,
        )

print("process_dataset() defined.")


process_dataset() defined.


## 10 · Analysis & Visualisation Helpers

In [25]:
SCALE_COLORS = {
    "unscaled": "#e74c3c",
    "median":   "#27ae60",
    "mean":     "#2980b9",
    "affine":   "#8e44ad",
}
SCALE_LABELS = {
    "unscaled": "Unscaled",
    "median":   "Median Scale",
    "mean":     "Mean Scale",
    "affine":   "Affine Scale",
}


def load_results() -> pd.DataFrame:
    df = pd.read_csv(OUTPUT_CSV_PATH)
    df = df.dropna(subset=[
        "unscaled_rmse", "median_rmse", "mean_rmse", "affine_rmse",
        "lidar_avg_depth", "lidar_min_depth",
    ])
    return df


def _save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    os.makedirs(PLOT_DIR, exist_ok=True)
    path = os.path.join(PLOT_DIR, name)
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    log.info("  ✓  %s", name)

print("Analysis helpers defined.")


Analysis helpers defined.


## 11 · Plot Functions

In [26]:
# ── Plot 01: GT vs Predicted Depth ────────────────────────────────────────────
def plot_gt_vs_pred(df: pd.DataFrame) -> None:
    variants = [
        ("Unscaled",     "lidar_avg_depth", "unscaled_avg_depth"),
        ("Median Scale", "lidar_avg_depth", "median_avg_depth"),
        ("Mean Scale",   "lidar_avg_depth", "mean_avg_depth"),
        ("Affine Scale", "lidar_avg_depth", "affine_avg_depth"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, xcol, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x=xcol, y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_title(f"DA2 Metric  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH); ax.legend(fontsize=9)
    fig.suptitle("Ground-Truth vs Predicted Depth — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "01_gt_vs_pred_all_scales.png")


# ── Plot 02: GT Depth vs RMSE ─────────────────────────────────────────────────
def plot_gt_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DA2 Metric  |  {title}", fontsize=12)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("RMSE (m)")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, ymax)
    fig.suptitle("GT Depth vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "02_gt_vs_rmse_all_scales.png")


# ── Plot 03: Horizontal Angle vs RMSE ────────────────────────────────────────
def plot_angle_vs_rmse(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    ymax = df[[v[1] for v in variants]].max().max() * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); axes = axes.flatten()
    for ax, (title, ycol), color in zip(axes, variants, SCALE_COLORS.values()):
        sns.regplot(ax=ax, data=df, x="horizontal_angle", y=ycol,
                    scatter_kws={"s": 25, "alpha": 0.35, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.set_title(f"DA2 Metric  |  {title}: Angle vs RMSE", fontsize=12)
        ax.set_xlabel("Horizontal Angle (°)"); ax.set_ylabel("RMSE (m)"); ax.set_ylim(0, ymax)
    fig.suptitle("Horizontal Angle vs RMSE — all scale variants", fontsize=15, y=1.01)
    plt.tight_layout(); _save_fig(fig, "03_angle_vs_rmse_all_scales.png")


# ── Plot 04: Scale Method Comparison Bar Chart ────────────────────────────────
def plot_scale_comparison_bars(df: pd.DataFrame) -> None:
    metrics_spec = [
        ("RMSE (m)  ↓", ["unscaled_rmse",   "median_rmse",   "mean_rmse",   "affine_rmse"]),
        ("Abs Rel   ↓", ["unscaled_abs_rel", "median_abs_rel","mean_abs_rel","affine_abs_rel"]),
        ("δ < 1.25  ↑", ["unscaled_delta1",  "median_delta1", "mean_delta1", "affine_delta1"]),
    ]
    labels = ["Unscaled","Median","Mean","Affine"]
    colors = list(SCALE_COLORS.values())
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, (metric_name, cols) in zip(axes, metrics_spec):
        vals = [df[c].mean() for c in cols]
        bars = ax.bar(labels, vals, color=colors, edgecolor="white", width=0.6)
        ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=10)
        ax.set_title(metric_name, fontsize=13); ax.set_ylabel(metric_name)
        ax.set_ylim(0, max(vals) * 1.22); ax.grid(axis="y", alpha=0.35)
    fig.suptitle("DA2 Metric — Scale Method Comparison", fontsize=15, y=1.02)
    plt.tight_layout(); _save_fig(fig, "04_scale_comparison_bars.png")


# ── Plot 05: Model vs LiDAR by Object Category ───────────────────────────────
def plot_model_vs_lidar_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    n = len(cats)
    if n == 0: return
    fig, axes = plt.subplots(1, max(n, 1), figsize=(7 * n, 6))
    if n == 1: axes = [axes]
    for ax, cat in zip(axes, cats):
        sub = df[df["super_cat"] == cat]
        for key in ["unscaled","median","mean","affine"]:
            ax.scatter(sub["lidar_avg_depth"], sub[f"{key}_avg_depth"],
                       s=18, alpha=0.4, c=SCALE_COLORS[key], label=SCALE_LABELS[key])
        ax.plot([0, MAX_DEPTH], [0, MAX_DEPTH], "k--", linewidth=1.5, label="Ideal 1:1")
        ax.set_xlim(0, MAX_DEPTH); ax.set_ylim(0, MAX_DEPTH)
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Predicted Depth (m)")
        ax.set_title(f"Category: {cat}", fontsize=12); ax.legend(fontsize=8)
    fig.suptitle("Predicted Depth vs LiDAR GT — by Object Category", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "05_model_vs_lidar_by_category.png")


# ── Plot 06: RMSE Distribution Histograms ─────────────────────────────────────
def plot_rmse_distributions(df: pd.DataFrame) -> None:
    variants = [("Unscaled","unscaled_rmse"),("Median Scale","median_rmse"),
                ("Mean Scale","mean_rmse"),("Affine Scale","affine_rmse")]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10)); axes = axes.flatten()
    for ax, (label, col), color in zip(axes, variants, SCALE_COLORS.values()):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(vals.mean(), color="black", linestyle="--", linewidth=1.8,
                   label=f"Mean   = {vals.mean():.3f} m")
        ax.axvline(vals.median(), color="grey", linestyle=":", linewidth=1.8,
                   label=f"Median = {vals.median():.3f} m")
        ax.set_title(f"RMSE Distribution — {label}", fontsize=12)
        ax.set_xlabel("RMSE (m)"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("RMSE Distributions Across All Masks", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "06_rmse_distributions.png")


# ── Plot 07: Scale Factor Distributions ──────────────────────────────────────
def plot_scale_distributions(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        vals = df[col].dropna()
        ax.hist(vals, bins=60, color=color, alpha=0.72, edgecolor="white")
        ax.axvline(1.0, color="black", linestyle="--", linewidth=1.8, label="Scale = 1.0")
        ax.axvline(vals.mean(), color="red", linestyle=":", linewidth=1.8,
                   label=f"Mean = {vals.mean():.3f}")
        ax.set_title(f"{label} Distribution", fontsize=12)
        ax.set_xlabel("Scale Factor"); ax.set_ylabel("Count"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor Distributions", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "07_scale_distributions.png")


# ── Plot 08: 3D Error Surfaces ────────────────────────────────────────────────
def plot_3d_error_surfaces(df: pd.DataFrame, degree: int = 2) -> None:
    variants = [("Unscaled","unscaled_rmse","b"),("Median Scale","median_rmse","g"),
                ("Mean Scale","mean_rmse","orange"),("Affine Scale","affine_rmse","purple")]
    xcol, ycol = "lidar_avg_depth", "horizontal_angle"
    df_c = df.dropna(subset=[xcol, ycol] + [v[1] for v in variants])
    x_min, x_max = df_c[xcol].min(), df_c[xcol].max()
    y_min, y_max = df_c[ycol].min(), df_c[ycol].max()
    z_min = df_c[[v[1] for v in variants]].min().min()
    z_max = df_c[[v[1] for v in variants]].max().max()
    buf = lambda r: r * 0.08
    fig = plt.figure(figsize=(26, 6))
    fig.suptitle(f"DA2 Metric — 3D Error Analysis (poly deg={degree})", fontsize=14)
    for sp_idx, (label, zcol, color) in enumerate(variants, start=1):
        ax = fig.add_subplot(1, 4, sp_idx, projection="3d")
        ax.scatter(df_c[xcol], df_c[ycol], df_c[zcol], c=color, alpha=0.35, s=8)
        Xf = df_c[[xcol, ycol]].values; Zf = df_c[zcol].values
        pipe = make_pipeline(PolynomialFeatures(degree), LinearRegression())
        pipe.fit(Xf, Zf)
        xs, ys = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
        zs = pipe.predict(np.c_[xs.ravel(), ys.ravel()]).reshape(xs.shape)
        ax.plot_surface(xs, ys, zs, color=color, alpha=0.22, rstride=1, cstride=1)
        ax.set_xlabel("GT Depth (m)", fontsize=8); ax.set_ylabel("H. Angle (°)", fontsize=8)
        ax.set_zlabel("RMSE (m)", fontsize=8); ax.set_title(label, fontsize=11)
        bx, by, bz = buf(x_max-x_min), buf(y_max-y_min), buf(z_max-z_min)
        ax.set_xlim(x_min-bx, x_max+bx); ax.set_ylim(y_min-by, y_max+by); ax.set_zlim(z_min-bz, z_max+bz)
    plt.tight_layout(); _save_fig(fig, "08_3d_error_surfaces.png")


# ── Plot 09: Depth-Bin Bias Analysis ─────────────────────────────────────────
def plot_depth_bin_analysis(df: pd.DataFrame) -> None:
    bins   = np.arange(0, MAX_DEPTH + 10, 10)
    labels = [f"{int(b)}–{int(b+10)}m" for b in bins[:-1]]
    df2    = df.copy()
    df2["depth_bin"] = pd.cut(df2["lidar_avg_depth"], bins=bins, labels=labels)
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    ax = axes[0]
    for key in ["unscaled","median","mean","affine"]:
        agg = df2.groupby("depth_bin", observed=True)[f"{key}_rmse"].mean()
        ax.plot(agg.index, agg.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE by Depth Bin — Scale Comparison")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    ax = axes[1]
    for key in ["unscaled","median","mean","affine"]:
        agg_pred  = df2.groupby("depth_bin", observed=True)[f"{key}_avg_depth"].mean()
        agg_lidar = df2.groupby("depth_bin", observed=True)["lidar_avg_depth"].mean()
        bias = agg_pred - agg_lidar
        ax.plot(bias.index, bias.values, "o-", color=SCALE_COLORS[key],
                label=SCALE_LABELS[key], linewidth=2)
    ax.axhline(0, color="black", linestyle="--", linewidth=1.5, label="Zero bias")
    ax.set_xlabel("Depth Bin"); ax.set_ylabel("Prediction Bias (m)")
    ax.set_title("Depth Bias by Bin (Pred − LiDAR)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)
    fig.suptitle("Depth-Bin Analysis — DA2 Metric", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "09_depth_bin_analysis.png")


# ── Plot 10: RMSE per Object Category ────────────────────────────────────────
def plot_rmse_by_category(df: pd.DataFrame) -> None:
    cats = sorted(df["super_cat"].dropna().unique())
    if len(cats) < 2: return
    keys, n_keys = ["unscaled","median","mean","affine"], 4
    x, w = np.arange(len(cats)), 0.18
    fig, ax = plt.subplots(figsize=(13, 6))
    for i, key in enumerate(keys):
        vals   = [df[df["super_cat"] == c][f"{key}_rmse"].mean() for c in cats]
        offset = (i - (n_keys - 1) / 2.0) * w
        bars   = ax.bar(x + offset, vals, width=w,
                        label=SCALE_LABELS[key], color=SCALE_COLORS[key])
        ax.bar_label(bars, fmt="%.2f", fontsize=8, padding=2)
    ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=11)
    ax.set_ylabel("Mean RMSE (m)")
    ax.set_title("RMSE per Object Category — DA2 Metric", fontsize=13)
    ax.legend(); ax.grid(axis="y", alpha=0.35)
    plt.tight_layout(); _save_fig(fig, "10_rmse_by_category.png")


# ── Plot 11: Scale Factor vs Depth ───────────────────────────────────────────
def plot_scale_vs_depth(df: pd.DataFrame) -> None:
    spec = [("median_scale","Median Scale",SCALE_COLORS["median"]),
            ("mean_scale","Mean Scale",SCALE_COLORS["mean"]),
            ("affine_scale","Affine Scale",SCALE_COLORS["affine"])]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (col, label, color) in zip(axes, spec):
        sns.regplot(ax=ax, data=df, x="lidar_avg_depth", y=col,
                    scatter_kws={"s": 18, "alpha": 0.30, "color": color},
                    line_kws={"color": "red", "linewidth": 2}, lowess=True)
        ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Scale = 1.0")
        ax.set_xlabel("LiDAR GT Depth (m)"); ax.set_ylabel("Scale Factor")
        ax.set_title(f"{label} vs Depth"); ax.legend(fontsize=9)
    fig.suptitle("Scale Factor vs Ground-Truth Depth", fontsize=14)
    plt.tight_layout(); _save_fig(fig, "11_scale_vs_depth.png")


# ── Plot 00: Summary Table ────────────────────────────────────────────────────
def plot_summary_table(df: pd.DataFrame) -> None:
    rows_data, specs = [], [
        ("Unscaled",     "unscaled_rmse",  "unscaled_abs_rel",  "unscaled_delta1",  None,           None),
        ("Median Scale", "median_rmse",    "median_abs_rel",    "median_delta1",    "median_scale", None),
        ("Mean Scale",   "mean_rmse",      "mean_abs_rel",      "mean_delta1",      "mean_scale",   None),
        ("Affine Scale", "affine_rmse",    "affine_abs_rel",    "affine_delta1",    "affine_scale", "affine_shift"),
    ]
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        rows_data.append([label, f"{sv:.4f}", f"{shv:.4f}" if shc else "—",
                          f"{df[rc].mean():.4f}", f"{df[ac].mean():.4f}",
                          f"{df[dc].mean():.4f}"])
    fig, ax = plt.subplots(figsize=(14, 3)); ax.axis("off")
    tbl = ax.table(cellText=rows_data,
                   colLabels=["Scale Method","Avg Scale","Avg Shift",
                               "Avg RMSE (m)","Avg Abs Rel","Avg δ<1.25"],
                   cellLoc="center", loc="center", bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1.2, 1.9)
    for j in range(6):
        tbl[0, j].set_facecolor("#2c3e50")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(rows_data) + 1):
        bg = "white" if i % 2 == 0 else "#f0f4f8"
        for j in range(6): tbl[i, j].set_facecolor(bg)
    fig.suptitle("DA2 Metric-Outdoor-Large — Scale Method Summary",
                 fontsize=14, fontweight="bold", y=1.04)
    plt.tight_layout(); _save_fig(fig, "00_summary_table.png", dpi=180)


def generate_all_plots(df: pd.DataFrame) -> None:
    """Generate and save all 12 analysis plots."""
    sns.set_style("whitegrid"); sns.set_context("paper", font_scale=1.1)
    log.info("Generating analysis plots → %s", PLOT_DIR)
    plot_summary_table(df);             plot_gt_vs_pred(df)
    plot_gt_vs_rmse(df);                plot_angle_vs_rmse(df)
    plot_scale_comparison_bars(df);     plot_model_vs_lidar_by_category(df)
    plot_rmse_distributions(df);        plot_scale_distributions(df)
    plot_3d_error_surfaces(df);         plot_depth_bin_analysis(df)
    plot_rmse_by_category(df);          plot_scale_vs_depth(df)
    log.info("All 12 plots saved to %s", PLOT_DIR)

print("All plot functions defined.")


All plot functions defined.


## 12 · Terminal Summary Printer

In [27]:
def print_summary(df: pd.DataFrame) -> None:
    W = 110
    print("\n" + "═" * W)
    print("  DA2 Metric-Outdoor-Large — Evaluation Summary")
    print(f"  Masks evaluated : {len(df):,}   |   Frames: {df['img_id'].nunique():,}")
    print("═" * W)
    hdr = (f"{'Scale Method':<24}"
           f"{'Avg Scale':>12}{'Avg Shift':>12}"
           f"{'RMSE (m)':>12}{'Abs Rel':>12}"
           f"{'δ<1.25':>10}{'LiDAR avg':>12}{'LiDAR min':>12}")
    print(hdr); print("─" * W)
    specs = [
        ("Unscaled",     "unscaled_rmse",  "unscaled_abs_rel",  "unscaled_delta1",  None,           None),
        ("Median Scale", "median_rmse",    "median_abs_rel",    "median_delta1",    "median_scale", None),
        ("Mean Scale",   "mean_rmse",      "mean_abs_rel",      "mean_delta1",      "mean_scale",   None),
        ("Affine Scale", "affine_rmse",    "affine_abs_rel",    "affine_delta1",    "affine_scale", "affine_shift"),
    ]
    for label, rc, ac, dc, sc, shc in specs:
        sv  = df[sc].mean()  if sc  else 1.0
        shv = df[shc].mean() if shc else 0.0
        print(
            f"  {label:<22}"
            f"{sv:>12.4f}{shv:>12.4f}"
            f"{df[rc].mean():>12.4f}{df[ac].mean():>12.4f}"
            f"{df[dc].mean():>10.4f}"
            f"{df['lidar_avg_depth'].mean():>12.4f}"
            f"{df['lidar_min_depth'].mean():>12.4f}"
        )
    print("═" * W + "\n")

print("print_summary() defined.")


print_summary() defined.


## 13 · Run Inference  *(skip if CSV already exists)*

Set `SKIP_INFERENCE = True` to jump straight to analysis from an existing CSV.


In [ ]:
# ── Set SKIP_INFERENCE = True to go straight to analysis from an existing CSV ─
SKIP_INFERENCE = True

if not SKIP_INFERENCE:
    log.info("── STEP 1/3: Indexing KITTI Eigen Split dataset …")
    val_list = index_kitti(use_train=USE_TRAIN)

    log.info("── STEP 2/3: Loading DA2 Metric-Outdoor-Large model …")
    da2_proc, da2_mdl = load_da2()
    # No SAM3 model to load — masks are read directly from MASK_ROOT.

    log.info("── STEP 3/3: Processing dataset (masks from %s) …", MASK_ROOT)
    process_dataset(da2_proc, da2_mdl, val_list)
else:
    log.info("Skipping inference — will use existing CSV at %s", OUTPUT_CSV_PATH)


15:14:51  INFO      Skipping inference — will use existing CSV at /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/DA2_Metric_Large_outputs/da2_metric_results.csv


## 14 · Analysis & Plots

Set `SKIP_PLOTS = True` to only print the terminal summary without regenerating  
all 12 plot files.


In [29]:
# ── Set SKIP_PLOTS = True to only print summary without regenerating plots ─────
SKIP_PLOTS = False

if os.path.exists(OUTPUT_CSV_PATH):
    log.info("Loading results CSV …")
    df = load_results()
    print_summary(df)

    if not SKIP_PLOTS:
        generate_all_plots(df)
else:
    log.warning(
        "Results CSV not found at %s. "
        "Run the inference cell first (SKIP_INFERENCE = False).",
        OUTPUT_CSV_PATH,
    )


15:14:51  INFO      Loading results CSV …
15:14:51  INFO      Generating analysis plots → /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/DA2_Metric_Large_outputs/plots



══════════════════════════════════════════════════════════════════════════════════════════════════════════════
  DA2 Metric-Outdoor-Large — Evaluation Summary
  Masks evaluated : 8,205   |   Frames: 645
══════════════════════════════════════════════════════════════════════════════════════════════════════════════
Scale Method               Avg Scale   Avg Shift    RMSE (m)     Abs Rel    δ<1.25   LiDAR avg   LiDAR min
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  Unscaled                    1.0000      0.0000      6.6252      0.2343    0.6770     31.3552     26.5831
  Median Scale                0.9344      0.0000      2.8755      0.0639    0.9472     31.3552     26.5831
  Mean Scale                  0.9311      0.0000      2.6649      0.0666    0.9495     31.3552     26.5831
  Affine Scale                0.4699     15.7007      1.7467      0.0393    0.9767     31.3552     26.5831
═══════════════════════════════════════

15:14:52  INFO        ✓  00_summary_table.png
15:15:10  INFO        ✓  01_gt_vs_pred_all_scales.png
15:15:28  INFO        ✓  02_gt_vs_rmse_all_scales.png
15:15:48  INFO        ✓  03_angle_vs_rmse_all_scales.png
15:15:49  INFO        ✓  04_scale_comparison_bars.png
15:15:51  INFO        ✓  05_model_vs_lidar_by_category.png
15:15:53  INFO        ✓  06_rmse_distributions.png
15:15:54  INFO        ✓  07_scale_distributions.png
15:15:59  INFO        ✓  08_3d_error_surfaces.png
15:16:00  INFO        ✓  09_depth_bin_analysis.png
15:16:00  INFO        ✓  10_rmse_by_category.png
15:16:16  INFO        ✓  11_scale_vs_depth.png
15:16:16  INFO      All 12 plots saved to /STORAGE/Arka/a_MDE_Code/1_Anaysis/Output/DA2_Metric_Large_outputs/plots
